In [0]:
-- ============================================================
-- Script : fix_fact_product_sk
-- Purpose: One-time repair of fact_order_items.product_sk.
--
-- Root cause: fact_order_items.product_sk currently holds values
-- in the range 1-57 (10 distinct), while dim_product.product_sk
-- is currently in the range 65-72 (8 distinct) - a completely
-- disjoint range. None of the 34 fact rows reference a product_sk
-- that exists in the current dim_product table.
--
-- Most likely explanation: dim_product was rebuilt/reloaded at
-- some point after the fact rows were originally inserted, and
-- its surrogate key sequence produced new values. The idempotent
-- MERGE in 05_load_facts only INSERTs new (unmatched) order items -
-- it never re-checks or updates product_sk on rows that are already
-- present, so the stale values were never corrected.
--
-- This script re-derives the correct, CURRENT product_sk for every
-- fact row by joining back through stg_order_items.product_id
-- (the stable natural/source key) to the current dim_product table,
-- and updates any row whose stored product_sk no longer matches.
-- ============================================================

-- 1) Diagnose (optional, run first to see what will change)
SELECT
    f.order_item_id,
    f.product_sk        AS current_product_sk,
    dp.product_sk        AS correct_product_sk
FROM brewflow_warehouse.fact_order_items f
JOIN brewflow_staging.stg_order_items   oi ON f.order_item_id = oi.order_item_id
JOIN brewflow_warehouse.dim_product     dp ON oi.product_id  = dp.product_id
WHERE f.product_sk <> dp.product_sk;

-- 2) Repair: update every row whose product_sk is stale
MERGE INTO brewflow_warehouse.fact_order_items AS target
USING (
    SELECT
        oi.order_item_id,
        dp.product_sk AS correct_product_sk
    FROM brewflow_staging.stg_order_items oi
    JOIN brewflow_warehouse.dim_product   dp ON oi.product_id = dp.product_id
) AS source
ON target.order_item_id = source.order_item_id
WHEN MATCHED AND target.product_sk <> source.correct_product_sk THEN
    UPDATE SET target.product_sk = source.correct_product_sk;

-- 3) Validate: no fact row should reference a non-existent product_sk anymore
SELECT COUNT(*) AS rows_with_invalid_product_sk
FROM brewflow_warehouse.fact_order_items f
LEFT JOIN brewflow_warehouse.dim_product dp ON f.product_sk = dp.product_sk
WHERE dp.product_sk IS NULL;
-- Erwartet: 0

-- 4) Re-check revenue by category (should now sum to the full total again)
SELECT dp.category, SUM(f.revenue) AS total_revenue
FROM brewflow_warehouse.fact_order_items f
JOIN brewflow_warehouse.dim_product dp ON f.product_sk = dp.product_sk
GROUP BY dp.category
ORDER BY total_revenue DESC;